# Transaction Foundation Model on Ray — Part 9: Run the whole pipeline on Anyscale

<div align="left">
  <a target="_blank" href="https://console.anyscale.com/template-preview/fintech_transaction_fm"><img src="https://img.shields.io/badge/🚀 Run_on-Anyscale-9hf"></a>&nbsp;
  <a href="https://github.com/anyscale/templates/tree/main/templates/fintech_transaction_fm" role="button"><img src="https://img.shields.io/static/v1?label=&message=View%20On%20GitHub&color=586069&logo=github&labelColor=2f363d"></a>&nbsp;
</div>

**⏱️ Time to complete**: ~10 min

---

Parts 2–7 ran each stage interactively so you could see the pattern. In production the whole thing runs headless, on a schedule, as a single command — and because every stage is Ray, that command is one Ray program on one autoscaling cluster. This notebook runs it end to end, then shows how to submit it as an Anyscale Job.

## One command, one fused pipeline

`scripts/run_pipeline.py` runs all six stages — data → tokenize → pretrain → embed → downstream → validate — in **one Ray driver**. The step-by-step notebooks wrote Parquet at every boundary so each stage was independently runnable; the production pipeline instead keeps the intermediates in the cluster:

- the tokenized **pretrain windows** are materialized in the object store and handed to Ray Train as a live dataset — no Parquet round-trip, and each epoch iterates from memory;
- the tokenized **eval windows** stream straight from the CPU tokenizer tasks into the GPU embedding actors in one Ray Data topology — they never touch disk.

Only the durable artifacts hit storage: the raw data, the vocab, the model checkpoint, the embeddings (the product), and the downstream metrics. The run ends with `validate`, which asserts every stage produced sane artifacts — the same check CI runs.

In [ ]:
# The entire pipeline, end to end, at mini scale (CPU). Same command the
# Anyscale Job runs — only the --scale value changes.
!python scripts/run_pipeline.py --scale mini

## Submit it as an Anyscale Job

The same script is the entrypoint in `job_config.yaml`, so the run above becomes a scheduled, autoscaling job by changing nothing but the scale:

```bash
anyscale job submit --config-file job_config.yaml                              # entrypoint: --scale small
anyscale job submit --config-file job_config.yaml -- python scripts/run_pipeline.py --scale full
```

The job's compute config declares the cluster shape, and the autoscaler fills it on demand:

- a **GPU worker group** (`g4dn.xlarge`, 0→4 nodes) that spins up only for the pretrain and embed stages and scales back to zero when idle — single-GPU nodes bin-pack exactly to the `num_workers` each scale asks for;
- a **CPU worker group** for the 24M-row tokenize shuffle, so GPU nodes never do shuffle work;
- `max_retries: 1`, and the pipeline wipes stale stage outputs on entry so a retry is idempotent (Ray's Parquet writes *append*, so a half-finished attempt would otherwise double a dataset).

Going from `mini` (this CPU run) to `full` (multi-GPU) is the `--scale` flag — the program is identical.

## Observability & fault tolerance

Because every stage is Ray on one cluster, one set of tools covers the whole pipeline.

**Observability** (in the Anyscale workspace/job UI)
- **Ray Dashboard** — watch batches stream through the tokenize and embed pipelines, per-operator throughput, object-store usage, and GPU utilization during pretraining.
- **Ray Train** reports per-epoch MLM metrics and records checkpoint lineage.
- **Ray Serve** exposes per-replica latency, ongoing requests, and autoscaling events.

**Fault tolerance** (built in)
- **Ray Train** checkpoints to shared storage, so a lost worker resumes from the last checkpoint instead of restarting pretraining.
- **Ray Data** retries failed tasks per block — a transient failure re-runs one partition, not the whole stage.
- **Streaming + backpressure** bound memory across the fused pipeline, so a 24M-row run doesn't OOM the driver.
- The job's `max_retries` plus the fresh-artifacts-on-entry rule make a full job retry safe and idempotent.

> *(Fold your own screenshots / dashboard links in here — observability is best shown live.)*

## What you built

A transaction foundation model, end to end, as one Ray program on one cluster:

| Stage | Ray primitive | Scales by |
|-------|---------------|-----------|
| Load & tokenize | Ray Data `map_groups` | `shuffle_partitions`, cluster size |
| Pretrain (MLM) | Ray Train + DDP | `num_workers`, `use_gpu` |
| Batch embed | Ray Data `map_batches` (actor pool) | `num_workers`, `num_gpus` |
| Downstream fraud | Ray Train `XGBoostTrainer` | `num_workers`, `use_gpu` |
| Online serve | Ray Serve | replica autoscaling |

The model was never the hard part. Tokenizing petabytes of transactions, pretraining across GPUs, re-embedding every customer on a schedule, and serving it online — all in one framework, the same code from a laptop-sized `mini` run to a multi-node GPU cluster — is the story.

---

## Next

**Part 9 — Scaling up, and the bottlenecks you'll hit**: the honest version of "change one config and it scales." Going from `mini` to the full dataset, you hit real walls — data that won't fit a node, idle GPUs, disk-bound stage boundaries, latency under load — and Part 9 shows each one and its Ray/Anyscale fix.